In [1]:
# ================== MOUNT DRIVE ==================
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
import pandas as pd
import os
import random
import csv
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import Dataset, DataLoader
from PIL import Image
from collections import Counter
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import (f1_score, confusion_matrix, roc_curve, auc,
                             recall_score, classification_report)
from sklearn.preprocessing import label_binarize
from tqdm import tqdm
import timm
import seaborn as sns

# ========================== 1. CONFIG ==========================
SEEDS = [62]
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

BASE_DATA_DIR = "/content/drive/MyDrive/VĐHĐ_TTNT/Skin cancer ISIC The International Skin Imaging Collaboration"
TRAIN_DIR = os.path.join(BASE_DATA_DIR, "Train")
TEST_DIR  = os.path.join(BASE_DATA_DIR, "Test")

# THAY ĐỔI: thư mục lưu kết quả cho model mới
SAVE_ROOT = "/content/drive/MyDrive/VĐHĐ_TTNT/EfficientNetB0_CBAM_SwinSmall"
os.makedirs(SAVE_ROOT, exist_ok=True)

NUM_CLASSES       = 9
BATCH_SIZE        = 32
NUM_EPOCHS        = 50
LEARNING_RATE     = 3e-4
WEIGHT_DECAY      = 1e-4
T_MAX             = 20
OVERSAMPLE_FACTOR = 5
PATIENCE          = 6

label_map = {
    0: 'actinic keratosis',
    1: 'basal cell carcinoma',
    2: 'dermatofibroma',
    3: 'melanoma',
    4: 'nevus',
    5: 'pigmented benign keratosis',
    6: 'seborrheic keratosis',
    7: 'squamous cell carcinoma',
    8: 'vascular lesion'
}

# ========================== 2. DATA ==========================

train_dataset_raw = datasets.ImageFolder(TRAIN_DIR)
test_dataset      = datasets.ImageFolder(TEST_DIR)

class RandomAugmentationPerImage:
    def __init__(self):
        self.augmentations = [
            transforms.RandomHorizontalFlip(0.5),
            transforms.RandomVerticalFlip(0.5),
            transforms.RandomRotation(40),
            transforms.ColorJitter(0.2, 0.2, 0.2)
        ]
        self.resize    = transforms.Resize((224, 224))
        self.to_tensor = transforms.ToTensor()
        self.normalize = transforms.Normalize(
            [0.485, 0.456, 0.406],
            [0.229, 0.224, 0.225]
        )

    def __call__(self, img):
        img = self.resize(img)
        for aug in self.augmentations:
            if torch.rand(1) < 0.5:
                img = aug(img)
        img = self.to_tensor(img)
        img = self.normalize(img)
        return img

transform_train = RandomAugmentationPerImage()
transform_val   = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

class CustomDataset(Dataset):
    def __init__(self, paths, labels, transform=None):
        self.paths     = np.array(paths)
        self.labels    = np.array(labels)
        self.transform = transform

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):
        img   = Image.open(self.paths[idx]).convert('RGB')
        label = int(self.labels[idx])
        if self.transform:
            img = self.transform(img)
        return img, label

# ========================== 3. LOSS ==========================

class FocalLoss(nn.Module):
    def __init__(self, gamma=2, weight=None):
        super().__init__()
        self.gamma  = gamma
        self.weight = weight

    def forward(self, inputs, targets):
        ce = nn.CrossEntropyLoss(weight=self.weight, reduction='none')(inputs, targets)
        pt = torch.exp(-ce)
        return ((1 - pt) ** self.gamma * ce).mean()

# ========================== 4. EARLY STOPPING ==========================

class EarlyStopping:
    def __init__(self, patience=6):
        self.patience = patience
        self.best     = None
        self.counter  = 0
        self.stop     = False

    def __call__(self, metric):
        if self.best is None or metric > self.best:
            self.best    = metric
            self.counter = 0
        else:
            self.counter += 1
            if self.counter >= self.patience:
                self.stop = True

# ========================== 5. MODEL: EfficientNet-B0 + CBAM + Swin-Small ==========================

class ChannelAttention(nn.Module):
    def __init__(self, in_channels, reduction=16):
        super().__init__()
        self.avg = nn.AdaptiveAvgPool2d(1)
        self.max = nn.AdaptiveMaxPool2d(1)
        self.mlp = nn.Sequential(
            nn.Linear(in_channels, in_channels // reduction, bias=False),
            nn.ReLU(),
            nn.Linear(in_channels // reduction, in_channels, bias=False)
        )
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        b, c, _, _ = x.size()
        avg = self.mlp(self.avg(x).view(b, c))
        mx  = self.mlp(self.max(x).view(b, c))
        att = self.sigmoid(avg + mx).view(b, c, 1, 1)
        return x * att

class SpatialAttention(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv    = nn.Conv2d(2, 1, 7, padding=3, bias=False)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        avg = torch.mean(x, 1, keepdim=True)
        mx, _ = torch.max(x, 1, keepdim=True)
        x = torch.cat([avg, mx], 1)
        return self.sigmoid(self.conv(x))

class CBAM(nn.Module):
    def __init__(self, in_channels):
        super().__init__()
        self.ca = ChannelAttention(in_channels)
        self.sa = SpatialAttention()

    def forward(self, x):
        x = self.ca(x)
        return x * self.sa(x)

class CNNExtractorCBAM(nn.Module):
    """EfficientNet-B0 làm backbone trích xuất đặc trưng, kết hợp CBAM attention."""
    def __init__(self):
        super().__init__()
        self.backbone = timm.create_model('efficientnet_b0', pretrained=True, num_classes=0)
        self.cbam     = CBAM(1280)

    def forward(self, x):
        # [B, 3, 224, 224] → [B, 1280, 7, 7]
        x = self.backbone.forward_features(x)
        return self.cbam(x)   # [B, 1280, 7, 7]

class HybridSwin_CBAM(nn.Module):
    """
    EfficientNet-B0 + CBAM + Swin-Small Stage-2 Blocks

    Luồng dữ liệu:
        [B, 3, 224, 224]
            → CNNExtractorCBAM    → [B, 1280, 7, 7]
            → Upsample bilinear   → [B, 1280, 14, 14]   ← FIX: khớp với Swin stage-2
            → Conv1x1 Projection  → [B, 384, 14, 14]
            → Permute             → [B, 14, 14, 384]
            → Swin layers[2].blocks (dim=384, window=7 → 4 windows) → [B, 14, 14, 384]
            → LayerNorm → GAP → Linear → [B, 9]
    """
    def __init__(self):
        super().__init__()

        self.cnn      = CNNExtractorCBAM()  # output: [B, 1280, 7, 7]

        # FIX: upsample để khớp resolution mà Swin stage-2 kỳ vọng
        self.upsample = nn.Upsample(size=(14, 14), mode='bilinear', align_corners=False)

        # Projection: 1280 → 384 (dim của Swin-Small stage-2)
        self.proj = nn.Conv2d(1280, 384, kernel_size=1)

        swin = timm.create_model(
            'swin_small_patch4_window7_224',
            pretrained=True,
            num_classes=0,
            drop_path_rate=0.1
        )
        self.swin_blocks = swin.layers[2].blocks  # dim=384, expects 14×14
        self.norm        = nn.LayerNorm(384)
        self.head        = nn.Linear(384, NUM_CLASSES)

    def forward(self, x):
        x = self.cnn(x)                            # [B, 1280, 7, 7]
        x = self.upsample(x)                       # [B, 1280, 14, 14]
        x = self.proj(x)                           # [B, 384, 14, 14]
        x = x.permute(0, 2, 3, 1).contiguous()    # [B, 14, 14, 384]
        x = self.swin_blocks(x)                    # [B, 14, 14, 384]
        x = self.norm(x)
        x = x.mean(dim=(1, 2))                     # [B, 384]
        return self.head(x)                        # [B, 9]

# ========================== 6. MULTI SEED TRAINING ==========================

for SEED in SEEDS:
    print(f"\n========== SEED {SEED} ==========")
    random.seed(SEED)
    np.random.seed(SEED)
    torch.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)

    SEED_DIR = os.path.join(SAVE_ROOT, f"seed_{SEED}")
    os.makedirs(SEED_DIR, exist_ok=True)

    # Logging
    csv_file   = open(os.path.join(SEED_DIR, "training_log.csv"), "w", newline="")
    csv_writer = csv.writer(csv_file)
    header = ["epoch", "lr", "train_loss", "train_acc", "val_loss", "val_acc", "val_f1", "val_recall"]
    for i in range(NUM_CLASSES):
        header.append(f"{label_map[i]}_F1")
    csv_writer.writerow(header)

    txt_log = open(os.path.join(SEED_DIR, "training_log.txt"), "w")

    # Oversampling + Split
    targets = [train_dataset_raw.imgs[i][1] for i in range(len(train_dataset_raw.imgs))]
    class_counts   = Counter(targets)
    max_count      = max(class_counts.values()) * OVERSAMPLE_FACTOR
    weights        = {c: max_count / v for c, v in class_counts.items()}
    sample_weights = np.array([weights[t] for t in targets])
    indices = np.random.choice(
        len(targets),
        len(targets) * OVERSAMPLE_FACTOR,
        replace=True,
        p=sample_weights / sample_weights.sum()
    )

    oversampled   = [train_dataset_raw.imgs[i] for i in indices]
    paths, labels = zip(*oversampled)
    paths, labels = np.array(paths), np.array(labels)

    train_p, val_p, train_l, val_l = train_test_split(
        paths, labels,
        test_size=0.3,
        stratify=labels,
        random_state=SEED
    )

    train_loader = DataLoader(
        CustomDataset(train_p, train_l, transform_train),
        batch_size=BATCH_SIZE, shuffle=True,
        num_workers=2,     # load song song
        pin_memory=True
    )
    val_loader = DataLoader(
        CustomDataset(val_p, val_l, transform_val),
        batch_size=BATCH_SIZE, shuffle=False,
        num_workers=2,
        pin_memory=True
    )

    test_paths  = [p for p, _ in test_dataset.imgs]
    test_labels = [l for _, l in test_dataset.imgs]
    test_loader = DataLoader(
        CustomDataset(test_paths, test_labels, transform_val),
        batch_size=BATCH_SIZE, shuffle=False
    )

    # THAY ĐỔI: dùng HybridSwin_CBAM thay DenseNet121
    model = HybridSwin_CBAM().to(device)
    print(f"Model: HybridSwin_CBAM | Params: "
          f"{sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

    class_weights_arr = compute_class_weight(
        'balanced',
        classes=np.unique(train_l),
        y=train_l
    )
    class_weights_tensor = torch.tensor(class_weights_arr, dtype=torch.float32).to(device)

    criterion = FocalLoss(weight=class_weights_tensor)
    optimizer = optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=T_MAX)
    early     = EarlyStopping(PATIENCE)

    best_f1       = 0
    best_epoch    = 0
    best_class_f1 = None
    best_path     = os.path.join(SEED_DIR, "best_model.pt")

    # ---- Vòng lặp training ----
    scaler = torch.cuda.amp.GradScaler()
    for epoch in range(1, NUM_EPOCHS + 1):

        # --- TRAIN ---
        model.train()
        train_loss, correct, total = 0, 0, 0
        train_bar = tqdm(train_loader, desc="TRAIN", leave=True)

        for imgs, lbl in train_bar:
            imgs, lbl = imgs.to(device), lbl.to(device)
            optimizer.zero_grad()

            with torch.amp.autocast('cuda'):   # ← Thêm
                out  = model(imgs)
                loss = criterion(out, lbl)

            scaler.scale(loss).backward()      # ← Thay loss.backward()
            scaler.step(optimizer)             # ← Thay optimizer.step()
            scaler.update()                    # ← Thêm

            train_loss += loss.item()
            preds       = out.argmax(1)
            correct    += (preds == lbl).sum().item()
            total      += lbl.size(0)
            train_bar.set_postfix(
                loss=train_loss / (train_bar.n + 1),
                acc=correct / total
            )

        scheduler.step()
        lr         = scheduler.get_last_lr()[0]
        train_loss /= len(train_loader)
        train_acc   = correct / total

        # --- VALIDATION ---
        model.eval()
        val_loss, val_correct, val_total = 0, 0, 0
        val_preds, val_labels_all        = [], []
        val_bar = tqdm(val_loader, desc="VAL", leave=True)

        with torch.no_grad():
            for imgs, lbl in val_bar:
                imgs, lbl = imgs.to(device), lbl.to(device)
                out  = model(imgs)
                loss = criterion(out, lbl)

                val_loss    += loss.item()
                preds        = out.argmax(1)
                val_correct += (preds == lbl).sum().item()
                val_total   += lbl.size(0)

                val_preds.extend(preds.cpu().numpy())
                val_labels_all.extend(lbl.cpu().numpy())

                val_bar.set_postfix(
                    loss=val_loss / (val_bar.n + 1),
                    acc=val_correct / val_total
                )

        val_loss  /= len(val_loader)
        val_acc    = val_correct / val_total
        val_f1     = f1_score(val_labels_all, val_preds, average='macro')
        val_recall = recall_score(val_labels_all, val_preds, average='macro', zero_division=0)
        class_f1   = f1_score(val_labels_all, val_preds, average=None)

        summary = (
            f"Epoch {epoch}/{NUM_EPOCHS} | LR: {lr:.6f} | "
            f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f} | "
            f"Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f} | "
            f"Val F1: {val_f1:.4f}"
        )
        print(summary)
        txt_log.write(summary + "\n")

        row = [epoch, lr, train_loss, train_acc, val_loss, val_acc, val_f1, val_recall]  # ← thêm val_recall
        for i in range(NUM_CLASSES):
            print(f"   {label_map[i]} F1: {class_f1[i]:.4f}")
            txt_log.write(f"   {label_map[i]} F1: {class_f1[i]:.4f}\n")
            row.append(class_f1[i])

        csv_writer.writerow(row)

        if val_f1 > best_f1:
            best_f1       = val_f1
            best_epoch    = epoch
            best_class_f1 = class_f1.copy()
            torch.save(model.state_dict(), best_path)

        early(val_f1)
        if early.stop:
            print(f"[Early Stopping] Epoch {epoch}")
            break

    print("\n===== BEST EPOCH SUMMARY =====")
    print(f"Best Epoch: {best_epoch} | Val F1: {best_f1:.4f}")
    for i in range(NUM_CLASSES):
        print(f"   {label_map[i]} F1: {best_class_f1[i]:.4f}")

    csv_file.close()
    txt_log.close()

    # ========================== PLOT TRAINING CURVES ==========================
    df = pd.read_csv(os.path.join(SEED_DIR, "training_log.csv"))

    fig, axes = plt.subplots(1, 3, figsize=(18, 5))

    # --- Loss ---
    axes[0].plot(df["epoch"], df["train_loss"], label="Train", color="steelblue")
    axes[0].plot(df["epoch"], df["val_loss"],   label="Val",   color="orange")
    axes[0].set_title("Loss")
    axes[0].legend()
    axes[0].grid(alpha=0.3)

    # --- Accuracy ---
    axes[1].plot(df["epoch"], df["train_acc"], label="Train", color="steelblue")
    axes[1].plot(df["epoch"], df["val_acc"],   label="Val",   color="orange")
    axes[1].set_title("Accuracy")
    axes[1].legend()
    axes[1].grid(alpha=0.3)

    # --- Macro F1 & Recall (xấp xỉ qua mean class F1) ---
    f1_cols = [f"{label_map[i]}_F1" for i in range(NUM_CLASSES)]
    df["macro_recall"] = df[f1_cols].mean(axis=1)

    # Thay hoàn toàn phần vẽ Macro F1 & Recall
    axes[2].plot(df["epoch"], df["val_f1"],     label="F1",     color="steelblue")
    axes[2].plot(df["epoch"], df["val_recall"], label="Recall", color="orange")
    axes[2].set_title("Macro F1 & Recall")
    axes[2].legend()
    axes[2].grid(alpha=0.3)

    plt.tight_layout()
    curves_path = os.path.join(SEED_DIR, "curves.png")
    plt.savefig(curves_path)
    plt.show()
    plt.close()
    print(f"[✓] Đã lưu curves.png → {curves_path}")

    # ========================== TEST PHASE ==========================
    print(f"\n--- BẮT ĐẦU KIỂM THỬ CUỐI CÙNG SEED {SEED} ---")
    model.load_state_dict(torch.load(best_path))
    model.eval()

    preds_all, labels_all, probs_all = [], [], []

    with torch.no_grad():
        for imgs, lbl in tqdm(test_loader, desc=f"Final Test Seed {SEED}"):
            imgs = imgs.to(device)

            with torch.amp.autocast('cuda'):
                # TTA: 4 augmentation versions
                out1 = model(imgs)
                out2 = model(torch.flip(imgs, [3]))
                out3 = model(torch.flip(imgs, [2]))
                out4 = model(torch.rot90(imgs, 1, [2, 3]))

                p1 = torch.softmax(out1, dim=1)
                p2 = torch.softmax(out2, dim=1)
                p3 = torch.softmax(out3, dim=1)
                p4 = torch.softmax(out4, dim=1)

                mean_probs = (p1 + p2 + p3 + p4) / 4
                preds      = mean_probs.argmax(dim=1)

            preds_all.extend(preds.cpu().numpy())
            labels_all.extend(lbl.numpy())
            probs_all.extend(mean_probs.cpu().numpy())

    labels_all = np.array(labels_all)
    preds_all  = np.array(preds_all)
    probs_all  = np.array(probs_all)

    # 1. Confusion Matrix
    class_names = [label_map[i] for i in range(NUM_CLASSES)]
    cm = confusion_matrix(labels_all, preds_all)
    plt.figure(figsize=(12, 10))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                xticklabels=class_names, yticklabels=class_names)
    plt.title(f"Confusion Matrix - Seed {SEED} (TTA Enabled)")
    plt.ylabel('True Label')
    plt.xlabel('Predicted Label')
    plt.tight_layout()
    plt.savefig(os.path.join(SEED_DIR, "confusion_matrix.png"))
    plt.show()
    plt.close()

    # 2. ROC Curve
    y_bin = label_binarize(labels_all, classes=list(range(NUM_CLASSES)))
    plt.figure(figsize=(10, 8))
    for i in range(NUM_CLASSES):
        if np.sum(y_bin[:, i]) > 0:
            fpr, tpr, _ = roc_curve(y_bin[:, i], probs_all[:, i])
            roc_auc = auc(fpr, tpr)
            plt.plot(fpr, tpr, label=f'{label_map[i]} (AUC = {roc_auc:.3f})')

    plt.plot([0, 1], [0, 1], 'k--', alpha=0.5)
    plt.title(f'ROC Curves - Seed {SEED}')
    plt.legend(loc="lower right", fontsize=9)
    plt.grid(alpha=0.3)
    plt.savefig(os.path.join(SEED_DIR, "roc_curve.png"))
    plt.show()
    plt.close()

    # 3. Classification Report
    report = classification_report(labels_all, preds_all, target_names=class_names)
    print(f"\n===== RESULT SUMMARY SEED {SEED} =====")
    print(report)

    with open(os.path.join(SEED_DIR, "final_report.txt"), "w", encoding="utf-8") as f:
        f.write(f"Best Epoch: {best_epoch}\n")
        f.write(report)

    # Dọn dẹp bộ nhớ
    del model
    torch.cuda.empty_cache()

print("\n--- TOÀN BỘ QUÁ TRÌNH HOÀN TẤT ---")

Device: cuda

========== SEED 62 ==========


/tmp/ipykernel_4882/1288232098.py:328: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler()


Model: HybridSwin_CBAM | Params: 36,685,439


VAL: 100%|██████████| 105/105 [00:47<00:00,  2.20it/s, acc=0.645, loss=0.46]


Epoch 1/50 | LR: 0.000298 | Train Loss: 1.0051 | Train Acc: 0.4366 | Val Loss: 0.4602 | Val Acc: 0.6445 | Val F1: 0.6259
   actinic keratosis F1: 0.7554
   basal cell carcinoma F1: 0.3870
   dermatofibroma F1: 0.8559
   melanoma F1: 0.3049
   nevus F1: 0.6578
   pigmented benign keratosis F1: 0.5025
   seborrheic keratosis F1: 0.6942
   squamous cell carcinoma F1: 0.5229
   vascular lesion F1: 0.9525


VAL:   0%|          | 0/105 [00:00<?, ?it/s]


KeyboardInterrupt: 